Note: Write your code in the code cells, and your responses in markdown. 
Run the entire script and display the outputs of your code. 

Due: **11:59PM Central Time on Monday, 11/03**. Upload both your code (.ipynb) and responses (html or pdf) to Canvas by then. 

In [ ]:
# Packages you might need: (pip install ... if you don't have them)
import pandas as pd # for data manipulation
import os  # for setting directory 
print(os.getcwd())
# os.chdir() # input your personal directory where the dataset is saved
import statsmodels.formula.api as smf # for OLS regressions
import numpy as np  # to work with arrays (vectors/matrices)

# NOTE from 10/29
- i = family
- t = twin

# Princeton Twins Data
In this problem set we will fit a few models to the Princeton Twins Survey data. The data set is called twins.csv. The variables are:
- famid = family id variable
- t=1,2 for twin #1 or twin #2
- age = age (some observations have coded in part year values)
- educ = education
- oeduc = education of twin
- lw = log wage
- married = dummy 1 if married 0 if not
- omarried = dummy if twin is married
- female = 1 if female
- ofemale = 1 if twin is female.
- exp = “labor market experience” = age - educ - 6
- oexp = experience of twin

NOTE: all twins are one of two identical twins in the data set. So female=ofemale in all cases.

In [ ]:
# Load dataset (using pandas "pd")
twins = pd.read_csv("twins.csv") # add your own directory if necessary
print(twins.head(6))

# verify the data is unique at (famid, t) level, where t = 1,2 for twin #1 or #2
assert 0==twins[['famid','t']].duplicated().sum()

# verify twins have the same sex: 
assert (twins['female']==twins['ofemale']).all()

## 1. OLS
Estimate a simple model relating log wages to: education, experience, experience-squared, married status, and female. 

In [ ]:
# experience squared 
twins['exp2'] = twins['exp']**2

## 2. Separate models for men and women
Fit the same model separately for men and women. Does the model look different?
- Note: drop "female" from the regression because we are estimating the regression separately by gender. 

## 3. Add mean family marriage rate
Construct the mean family marriage rate for each person in the data set (i.e., the fraction of the twins that is married, which can be 0, 1/2, or 1).

### (a) 
Verify that when you regress marriage of a twin on the average fraction of the siblings who are married, you get a coefficient of 1.

### (b) 
Add mean fraction of siblings married to your gender-specific wage models from part 2. How does the addition of this variable affect the estimated coefficient on marriage. Give an interpretation of the patterns and how they differ between men and women. 

### (c)
Instead of controlling for the mean family marriage rate, estimate the gender-specific wage models from part 2 with a control for the other twin's marriage status (variable “omarried”). Compare the regression coefficients on (married, omarried) with the coefficient on (married, mean married) in (b). 

## 4. Three ways to get the within estimator: 
In lecture 8 we show three ways to get the within estimator - de-meaned relative to the mean, fixed effects, and control function. Let's run 3 regressions for **male** twins. 

### (a) 
De-mean variables relative to the mean in each family: replace $x_{it}$ by $x_{it}-\bar{x}_{i}$ where $x_{it}$ includes education, experience-squared, and married status, and replace $y_{it}$ by $y_{it}-\bar{y}_{i}$. Fit a regression of $(y_{it}-\bar{y}_{i})$ on $x_{it}-\bar{x}_{i}$. 
- Note we dropped experience from the regression. See how experience is defined. The demeaned experience = - demeaned education (correlation = -1). 

In [ ]:
keys = ['famid','t'] # twins data unique at (family, twin) level
select= ['lw','educ','exp','exp2','married']
# focus on families with male twins:
fam_means = twins.loc[twins['female']==0].groupby(['famid'])[select].mean().reset_index(drop=False)
fam_means.columns=['famid'] + [f'mean_{x}' for x in select]

# merge fam_means with twins (note you may have defined mean marriage rate already) by famid
male_twins = pd.merge(twins.loc[twins['female']==0, keys+select], fam_means, on=['famid'],how='left')

# compute the demeaned x and y: 
for z in select:
    male_twins[f'd_{z}'] = male_twins[z] - male_twins[f'mean_{z}'] 

# note: d_exp = - d_educ. exp is defined as age - educ - 6. Within each family, twins are of the same age, d_exp = exp - (age  - mean_educ -6) = - educ + mean_educ = - d_educ
print(male_twins[['d_exp','d_educ']].corr())
assert (abs(male_twins['d_exp'] + male_twins['d_educ'])<1e-12).all()

In [ ]:
# OLS for within: formula = "d_lw ~ d_educ + d_exp + d_exp2 + d_married" 


### (b) Fixed Effects
Fit a regression of y_{it} on x_{it} and family fixed effects. 
- *Note if you use PanelOLS from linearmodels, make sure to drop collinear variables before adding “entityeffects”. For example, educ and exp are collinear with each other conditional on family fixed effects (twins have the same age). We have dropped exp from the regression.*

In [ ]:
# may consider: 
#  pip install linearmodels 
#  from linearmodels import PanelOLS

### (c) Control Function
Fit a regression of $y_{it}$ on $x_{it}$ and means $\bar{x}_{i}$ in each family. Verify if the coefficients on $x_{it}$ are the same as in (a) and (b). 